A notebook to see if we can use llama to classify contract types.

In [1]:
from typing import Dict

import pandas as pd

from langchain.prompts import PromptTemplate
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel
from pydantic import Field

In [ ]:
class ContractType(BaseModel):
    """Model for outputting contract type."""
    contract_type: str = Field(description="Type of contract: ['unknown', 'permanent', 'temporary', 'zero_hours']")


prompt = """
    Given a sentence from a job advert, it is your job to determine the type of contract being offered. The contract type can be one of the following:
    - 'permanent' i.e. a permanent job
    - 'temporary' i.e. temporary, fixed-term, secondment, maternity cover and so on
    - 'zero_hours' i.e. casual work with no guaranteed hours
    - 'unknown' i.e. it is unclear what contract type, if any, the sentence refers to
    
    **Rules for classification**:
    - If a **fixed-term contract** is mentioned, label it **temporary**.
    - If a **role can be both temporary and permanent**, label it **temporary**.
    - If no clear contract type is stated, label it **unknown**.
    - If the contract mentions **"zero hours"**, label it **zero_hours**.
    
    Examples:
    [
        {{"sentence": "This is a permanent role", "contract_type": "permanent"}},
        {{"sentence": "long term opportunity", "contract_type": "permanent"}},
        {{"sentence": "This is an initial 6-month contract with the possibility of becoming permanent", "contract_type": "temporary"}},
        {{"sentence": "Contract duration 6 months", "contract_type": "temporary"}},
        {{"sentence": "You must have 6 months experience", "contract_type": "unknown"}},
        {{"sentence": "fixed-term contract", "contract_type": "temporary"}},
        {{"sentence": "fixed-term and permanent contracts available", "contract_type": "temporary"}},
        {{"sentence": "This is a maternity cover position", "contract_type": "temporary"}},
        {{"sentence": "secondment", "contract_type": "temporary"}},
        {{"sentence": "This is a zero hours contract", "contract_type": "zero_hours"}},
        {{"sentence": "No zero-hours contracts", "contract_type": "unknown"}},
        {{"sentence": "You will be contracted to 36 hours per week", "contract_type": "unknown"}},
        {{"sentence": "Working days and nights", "contract_type": "unknown"}},
        {{"sentence": "Offering you flexible work or alternatively fixed term placements", "contract_type": "temporary"}},
        {{"sentence": "Long term contract", "contract_type": "permanent"}},
        {{"sentence": "NHS staff bank", "contract_type": "zero_hours"}}
    ]
    
    \n
    Classify the following sentence:
    {sentence}
    \n
    
    Output format:
    {{"contract_type": "<one of: permanent, temporary, zero_hours, unknown>"}}
    
    \n
    Please return **only** this JSON object.
    \n
    """

parser = JsonOutputParser(pydantic_object=ContractType)

final_prompt = PromptTemplate(
    template=prompt,
    input_variables=["sentence"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

model = "llama3.2"

ollama_model = ChatOllama(model=model, temperature=0)

llm_chain = final_prompt | ollama_model | parser

In [3]:
test_sentences = ["They have shifts up to 6 weeks in advance which cover Days   Nights and Weekends.",
                  "6 month recent experience working in NHS is a must What you'll get in return  Earn a highly competitive hourly pay rate.",
                  "PD for a rolling 6-month project based in Cardiff.",
                  "This role is a permanent contract covering 37.5 hours across the week and requires the successful applicant to be flexible to respond to the demands",
                  "As a nurse on our bank you'll be on a zero hour contract",
                  "we have a contract role in Preston you won't want to miss!Job Description.",
                  "Role type Contract Duration 6 months Day rate £500 - £600 per day IR35",
                  "The role is Contract between October 2022 and December 2022 The hourly rate is negotiable Full or Part time role Experience skills required",
                  "Flexible work -Fixed term contracts and AD-Hoc work -Multiple payment",
                  "fixed-term contracts and permanent positions all around the UK"
                  ]

In [ ]:
for sent in test_sentences:
    print(llm_chain.invoke({"sentence": sent}))

In [5]:
eval_sample = pd.read_csv('contract_evaluation.csv')

In [ ]:
eval_sample.head()

In [ ]:
import json

classified_contract_types = []
errors = []
for idx, row in eval_sample.iterrows():
    try:
        result = llm_chain.invoke({"sentence": row['sentences_split']})
        print(idx, result['contract_type'])
        
        if isinstance(result, dict) and 'contract_type' in result:
            classified_contract_types.append((idx, result['contract_type']))
        else:
            print(f"⚠️ Unexpected output at index {idx}: {result}")
            errors.append((idx, row['sentences_split'], result))
    
    except json.JSONDecodeError as e:
        print(f"❌ JSON decoding failed at index {idx}: {str(e)}")
        errors.append((idx, row['sentences_split'], "JSONDecodeError"))

    except Exception as e:
        print(f"❌ Unexpected error at index {idx}: {str(e)}")
        errors.append((idx, row['sentences_split'], str(e)))
    

In [ ]:
errors

In [ ]:
classified_contract_types

In [ ]:
classified_df = pd.DataFrame(classified_contract_types, columns=["idx", "contract_type"])
classified_df.head()

In [14]:
eval_sample2 = eval_sample.reset_index()  # Ensure index is a column
classified_df = classified_df.rename(columns={"contract_type": "contract_type_llama"})
merged_df = eval_sample2.merge(classified_df, left_on="index", right_on="idx", how="left")


In [ ]:
merged_df.columns

In [ ]:
merged_df['contract_type_llama'].isna().sum()

In [ ]:
merged_df['contract_type_llama'].fillna('unknown', inplace=True)
merged_df['contract_type_llama'].isna().sum()

In [19]:
merged_df.to_csv('contract_type_llama_labelled_for_eval.csv', index=False)